# 🎓 Student Performance Prediction

This notebook contains the complete pipeline for predicting student performance.

### 1. Import Libraries

In [ ]:
# Import required libraries for data handling, visualization, and ML models

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

import pickle
import os

### 📂 2. Load Dataset

In [ ]:
# Load dataset from local CSV file (using relative path)
data_path = "../data/student_performance_prediction.csv"

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    # Display basic info
    print("Shape:", df.shape)
    print(df.head())
    print(df.info())
else:
    print(f"Error: File not found at {data_path}")

### 🧹 3. Data Cleaning

In [ ]:
# Clean column names (remove extra spaces)
df.columns = df.columns.str.strip()

# Drop unnecessary ID column if present
for col in df.columns:
    if "id" in col.lower():
        df = df.drop(col, axis=1)

# Handle missing values
df = df.dropna()

print("\nMissing Values:\n", df.isnull().sum())

### 📊 4. Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of target variable
sns.countplot(x='Passed', data=df)
plt.title("Pass vs Fail Distribution")
plt.show()

# Correlation heatmap (numerical features only)
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Feature Correlation")
plt.show()

### 🎯 5. Target Encoding

In [ ]:
# Convert target variable (Yes/No → 1/0)
df['Passed'] = df['Passed'].map({'Yes': 1, 'No': 0})

### 🔄 6. Feature Encoding

In [ ]:
# Convert categorical variables into numeric using one-hot encoding
df = pd.get_dummies(df, drop_first=True)

### ✂️ 7. Split Data

In [ ]:
# Define features (X) and target (y)
X = df.drop("Passed", axis=1)
y = df["Passed"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

### ⚖️ 8. Scaling

In [ ]:
# Scale features for better model performance (especially Logistic Regression)
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### 🤖 9. Train Models

In [ ]:
# Initialize models
lr = LogisticRegression(max_iter=1000)
rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Train models
lr.fit(X_train, y_train)
rf.fit(X_train, y_train)

### 📈 10. Evaluate Models

In [ ]:
# Make predictions
lr_pred = lr.predict(X_test)
rf_pred = rf.predict(X_test)

# Evaluate Logistic Regression
print("\n--- Logistic Regression ---")
print("Accuracy:", accuracy_score(y_test, lr_pred))
print("ROC-AUC:", roc_auc_score(y_test, lr_pred))
print(classification_report(y_test, lr_pred))

# Evaluate Random Forest
print("\n--- Random Forest ---")
print("Accuracy:", accuracy_score(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))

# Confusion Matrix
sns.heatmap(confusion_matrix(y_test, rf_pred), annot=True, fmt='d')
plt.title("Confusion Matrix - Random Forest")
plt.show()

### ⚙️ 11. Model Optimization

In [ ]:
# Hyperparameter tuning using GridSearchCV
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 5, 10]
}

grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3)
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

# Evaluate optimized model
best_model = grid.best_estimator_
best_pred = best_model.predict(X_test)

print("\n--- Optimized Random Forest ---")
print("Accuracy:", accuracy_score(y_test, best_pred))

# Cross-validation score (using original features for CV)
# Note: X needs to be processed similarly if used for CV outside of the pipe
# For simplicity, we'll use X_train/y_train here if preferred, or X if it's already encoded
scores = cross_val_score(best_model, X_train, y_train, cv=5)
print("Cross-validation accuracy:", scores.mean())

### 💾 12. Save Model

In [ ]:
# Save the trained optimized model
os.makedirs('../model', exist_ok=True)
pickle.dump(best_model, open("../model/student_model.pkl", "wb"))

print("Model saved successfully to ../model/student_model.pkl!")

### 🔮 13. Inference

In [ ]:
# Load model
if os.path.exists("../model/student_model.pkl"):
    model = pickle.load(open("../model/student_model.pkl", "rb"))

    # Example input (modify as per dataset features)
    # Note: Ensure input matches the feature count and scaling if used
    sample = np.array([[10, 85, 75, 1, 0, 0, 1, 0]])  # Example features mapping to encoded columns
    
    # Apply scaling to sample if the model was trained on scaled data
    sample_scaled = scaler.transform(sample)
    
    prediction = model.predict(sample_scaled)

    if prediction[0] == 1:
        print("Student is likely to PASS")
    else:
        print("Student is likely to FAIL")
else:
    print("Model file not found.")